<a href="https://colab.research.google.com/github/Komal5578/DWM/blob/main/Apriori_my_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APRIORI: frequent itemsets + rules, no library (paper-ready)
**Edit only the ✏️ DATA cell** (transactions, minimum support, minimum confidence).
Output shows C1 → L1 → C2 → L2 … exactly like the hand method, including **pruning**.

In [1]:
from itertools import combinations
import pandas as pd

## ✏️ DATA

In [2]:
transactions = {
 'T1': ['I1','I2','I5'],
 'T2': ['I2','I4'],
 'T3': ['I2','I3'],
 'T4': ['I1','I2','I4'],
 'T5': ['I1','I3'],
 'T6': ['I2','I3'],
 'T7': ['I1','I3'],
 'T8': ['I1','I2','I3','I5'],
 'T9': ['I1','I2','I3'],
}
MIN_SUP  = 2      # minimum support COUNT. If the question gives %, use e.g. 0.22 * len(transactions)
MIN_CONF = 0.60   # minimum confidence (60% = 0.60)

T = [frozenset(t) for t in transactions.values()]
N = len(T)
print('Number of transactions:', N, '| MIN_SUP count =', MIN_SUP, '(', round(MIN_SUP/N*100,1), '% )')

Number of transactions: 9 | MIN_SUP count = 2 ( 22.2 % )


In [3]:
def count(itemset):
    return sum(itemset <= t for t in T)          # in how many transactions is itemset a subset?

def show(itemset): return '{' + ', '.join(sorted(itemset)) + '}'

## Step 1: Level-wise search (C = candidates, L = frequent)

In [4]:
frequent = {}                                      # every frequent itemset -> support count
items = sorted({i for t in T for i in t})
C = [frozenset([i]) for i in items]                # C1
k = 1
while C:
    counts = {c: count(c) for c in C}
    print(f'--- C{k} (candidates) ---')
    print(pd.DataFrame({'Itemset':[show(c) for c in C], 'Support count':list(counts.values()),
                        'Frequent?':['yes' if v >= MIN_SUP else 'NO (pruned)' for v in counts.values()]}).to_string(index=False))
    L = {c: v for c, v in counts.items() if v >= MIN_SUP}
    print(f'--- L{k} (frequent) ---', [show(c) for c in L], '\n')
    frequent.update(L)
    # JOIN step: combine frequent k-itemsets into (k+1)-itemsets
    Lk = list(L)
    cand = {a | b for a in Lk for b in Lk if len(a | b) == k + 1}
    # PRUNE step: every k-subset of a candidate must itself be frequent (Apriori property)
    C = [c for c in cand if all(frozenset(s) in L for s in combinations(c, k))]
    C.sort(key=lambda s: sorted(s))
    k += 1
print('Total frequent itemsets:', len(frequent))

--- C1 (candidates) ---
Itemset  Support count Frequent?
   {I1}              6       yes
   {I2}              7       yes
   {I3}              6       yes
   {I4}              2       yes
   {I5}              2       yes
--- L1 (frequent) --- ['{I1}', '{I2}', '{I3}', '{I4}', '{I5}'] 

--- C2 (candidates) ---
 Itemset  Support count   Frequent?
{I1, I2}              4         yes
{I1, I3}              4         yes
{I1, I4}              1 NO (pruned)
{I1, I5}              2         yes
{I2, I3}              4         yes
{I2, I4}              2         yes
{I2, I5}              2         yes
{I3, I4}              0 NO (pruned)
{I3, I5}              1 NO (pruned)
{I4, I5}              0 NO (pruned)
--- L2 (frequent) --- ['{I1, I2}', '{I1, I3}', '{I1, I5}', '{I2, I3}', '{I2, I4}', '{I2, I5}'] 

--- C3 (candidates) ---
     Itemset  Support count Frequent?
{I1, I2, I3}              2       yes
{I1, I2, I5}              2       yes
--- L3 (frequent) --- ['{I1, I2, I3}', '{I1, I2, I5}'] 

T

## Step 2: All frequent itemsets

In [5]:
fi = pd.DataFrame([[show(s), len(s), c, round(c/N, 3)] for s, c in frequent.items()],
                  columns=['Itemset','Size','Support count','Support'])
fi.sort_values(['Size','Itemset']).reset_index(drop=True)

,Itemset,Size,Support count,Support
0,{I1},1,6,0.667
1,{I2},1,7,0.778
2,{I3},1,6,0.667
3,{I4},1,2,0.222
4,{I5},1,2,0.222
5,"{I1, I2}",2,4,0.444
6,"{I1, I3}",2,4,0.444
7,"{I1, I5}",2,2,0.222
8,"{I2, I3}",2,4,0.444
9,"{I2, I4}",2,2,0.222


## Step 3: Association rules  A → B
confidence = sup(A∪B) / sup(A)  |  lift = confidence / sup(B)

In [6]:
rules = []
for itemset, sup in frequent.items():
    if len(itemset) < 2: continue
    for r in range(1, len(itemset)):
        for A in map(frozenset, combinations(itemset, r)):
            B = itemset - A
            conf = sup / count(A)
            lift = conf / (count(B) / N)
            rules.append([show(A)+' -> '+show(B), f'{sup}/{count(A)}', round(conf, 3), round(lift, 3), 'STRONG' if conf >= MIN_CONF else 'rejected'])
R = pd.DataFrame(rules, columns=['Rule','Support count / sup(A)','Confidence','Lift','Result'])
R.sort_values('Confidence', ascending=False).reset_index(drop=True)

,Rule,Support count / sup(A),Confidence,Lift,Result
0,{I5} -> {I1},2/2,1.000,1.500,STRONG
1,"{I1, I5} -> {I2}",2/2,1.000,1.286,STRONG
2,{I5} -> {I2},2/2,1.000,1.286,STRONG
3,"{I5} -> {I1, I2}",2/2,1.000,2.250,STRONG
4,"{I2, I5} -> {I1}",2/2,1.000,1.500,STRONG
5,{I4} -> {I2},2/2,1.000,1.286,STRONG
6,{I1} -> {I3},4/6,0.667,1.000,STRONG
7,{I1} -> {I2},4/6,0.667,0.857,STRONG
8,{I3} -> {I1},4/6,0.667,1.000,STRONG
9,{I3} -> {I2},4/6,0.667,0.857,STRONG


In [7]:
print('STRONG RULES (confidence >=', MIN_CONF, '):')
R[R['Result']=='STRONG'].sort_values('Confidence', ascending=False).reset_index(drop=True)

STRONG RULES (confidence >= 0.6 ):


,Rule,Support count / sup(A),Confidence,Lift,Result
0,{I4} -> {I2},2/2,1.000,1.286,STRONG
1,{I5} -> {I1},2/2,1.000,1.500,STRONG
2,"{I2, I5} -> {I1}",2/2,1.000,1.500,STRONG
3,"{I1, I5} -> {I2}",2/2,1.000,1.286,STRONG
4,"{I5} -> {I1, I2}",2/2,1.000,2.250,STRONG
5,{I5} -> {I2},2/2,1.000,1.286,STRONG
6,{I3} -> {I1},4/6,0.667,1.000,STRONG
7,{I1} -> {I2},4/6,0.667,0.857,STRONG
8,{I3} -> {I2},4/6,0.667,0.857,STRONG
9,{I1} -> {I3},4/6,0.667,1.000,STRONG


**Viva:** Apriori property = *every subset of a frequent itemset is frequent* (so supersets of an infrequent set are pruned).
Support = freq/N · Confidence = sup(A∪B)/sup(A) · Lift > 1 positive correlation, = 1 independent, < 1 negative.
Weakness: many database scans + large candidate sets → FP-Growth.